In [2]:
# Cell 1: Setup (Gemini planner pipeline, independent)
import os
import re
import json
import time
import subprocess
from pathlib import Path
from datetime import datetime

from dotenv import load_dotenv
from google import genai
from google.genai import types

# Project root (run notebook from formalizer-scaling/src/planner_gen)
PROJECT_ROOT = Path.cwd().parents[1]

# Data / prompt / output paths
DATA_DIR = PROJECT_ROOT / "data"
PROMPT_PATH = PROJECT_ROOT / "prompts" / "llm_planner_instruction.txt"
DOMAIN_PDDL_PATH = DATA_DIR / "pddl" / "domain.pddl"

# New output root for direct LLM-as-planner pipeline
MODEL_DIRNAME = "gemini_3_flash"
OUTPUT_ROOT = PROJECT_ROOT / "output" / "llm_planner" / MODEL_DIRNAME
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

# VAL executable (adjust if your environment differs)
VAL_PATH = Path.home() / "tools" / "VAL" / "build" / "linux64" / "Release" / "bin" / "Validate"

# Model + API
MODEL_NAME = "gemini-3-flash-preview"
ENV_PATH = PROJECT_ROOT / ".env"
load_dotenv(ENV_PATH)

GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
if not GOOGLE_API_KEY:
  raise ValueError(f"GOOGLE_API_KEY not found in {ENV_PATH}")

client = genai.Client(api_key=GOOGLE_API_KEY)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("OUTPUT_ROOT:", OUTPUT_ROOT)
print("VAL exists:", VAL_PATH.exists())
print("Domain PDDL exists:", DOMAIN_PDDL_PATH.exists())
print("Prompt exists:", PROMPT_PATH.exists())
print("Model:", MODEL_NAME)

PROJECT_ROOT: /home/sagemaker-user/formalizer-scaling
OUTPUT_ROOT: /home/sagemaker-user/formalizer-scaling/output/llm_planner/gemini_3_flash
VAL exists: True
Domain PDDL exists: True
Prompt exists: True
Model: gemini-3-flash-preview


In [3]:
# Cell 2: Load problem IDs, planner instruction, and ground-truth domain PDDL
import pandas as pd

# Use all available problems by default; edit this list later for targeted runs
PROBLEM_IDS = [f"p{i:02d}" if i < 100 else f"p{i}" for i in range(1, 201)]

PLANNER_INSTRUCTION = PROMPT_PATH.read_text()
DOMAIN_PDDL_TEXT = DOMAIN_PDDL_PATH.read_text()

print("Num problems:", len(PROBLEM_IDS))
print("First/last:", PROBLEM_IDS[0], PROBLEM_IDS[-1])
print("Planner prompt chars:", len(PLANNER_INSTRUCTION))
print("Domain PDDL chars:", len(DOMAIN_PDDL_TEXT))

Num problems: 200
First/last: p01 p200
Planner prompt chars: 1294
Domain PDDL chars: 997
